In [3]:
import os
import numpy as np
import pandas as pd
import tensorflow as tf

from pathlib import Path
from datetime import datetime

from sklearn.model_selection import train_test_split
from sklearn.metrics import accuracy_score, f1_score, classification_report

from tensorflow.keras import Sequential, Input
from tensorflow.keras.layers import Embedding, SimpleRNN, Bidirectional, Dropout, Dense
from tensorflow.keras.optimizers import Adam
from tensorflow.keras.preprocessing.text import Tokenizer
from tensorflow.keras.preprocessing.sequence import pad_sequences

SEED = 88
np.random.seed(SEED)
tf.random.set_seed(SEED)
tf.get_logger().setLevel("ERROR")

PROJECT_ROOT = Path.cwd().parent if Path.cwd().name == \"evaluation\" else Path.cwd()\nDATA_PATH = PROJECT_ROOT / \"datasets/Pizza reviews.csv\"\nWEIGHTS_PATH = PROJECT_ROOT / \"models/final_rnn.weights.h5\"

VOCAB_SIZE = 1000
EMBEDDING_DIM = 32
FINAL_SEQUENCE_LENGTH = 15

SENTIMENT_CLASSES = ["Negative", "Mixed/Neutral", "Positive"]

print("Setup complete.")
print("TensorFlow version:", tf.__version__)
print("Data path:", DATA_PATH)
print("Weights path:", WEIGHTS_PATH)


Setup complete.
TensorFlow version: 2.20.0
Data path: datasets/Pizza reviews.csv
Weights path: final_rnn.weights.h5


In [4]:
weights_file = Path(WEIGHTS_PATH)

print("Exists:", weights_file.exists())

if weights_file.exists():
    print("Size MB:", round(weights_file.stat().st_size / (1024 * 1024), 2))
    print("Last modified:", datetime.fromtimestamp(weights_file.stat().st_mtime))
else:
    raise FileNotFoundError(f"Could not find weights file: {WEIGHTS_PATH}")


Exists: True
Size MB: 0.55
Last modified: 2026-05-17 10:50:58


In [5]:
# Load and clean dataset using the same logic as the final notebook
raw_df = pd.read_csv(DATA_PATH)

required_columns = ["Review", "Score", "Language"]
missing_columns = [col for col in required_columns if col not in raw_df.columns]

if missing_columns:
    raise ValueError(f"Missing required columns in dataset: {missing_columns}")

clean_df = raw_df.copy()

# Valid numeric score only
clean_df["Score_Numeric"] = pd.to_numeric(clean_df["Score"], errors="coerce")
valid_score_mask = (
    clean_df["Score_Numeric"].notna()
    & clean_df["Score_Numeric"].between(0, 10, inclusive="both")
)
clean_df = clean_df[valid_score_mask].copy()

# Keep only modelling languages
clean_df = clean_df[clean_df["Language"].isin(["English", "Malay"])].copy()

# Drop unused note column if present
note_column = "Are there ways for you to generate more data? Spliting up sentences, would that help?"
if note_column in clean_df.columns:
    clean_df = clean_df.drop(columns=[note_column])

# Standardise review text
clean_df["Review_Text"] = clean_df["Review"].astype(str).str.strip()
clean_df = clean_df[clean_df["Review_Text"] != ""].copy()

# Same 3-class target mapping
def assign_final_sentiment(score):
    if score <= 3:
        return "Negative"
    elif score <= 7:
        return "Mixed/Neutral"
    else:
        return "Positive"

sentiment_label_map = {
    "Negative": 0,
    "Mixed/Neutral": 1,
    "Positive": 2
}

clean_df["Sentiment"] = clean_df["Score_Numeric"].apply(assign_final_sentiment)
clean_df["Sentiment_Encoded"] = clean_df["Sentiment"].map(sentiment_label_map)

# Same duplicate handling before split
clean_df["Review_Normalized"] = (
    clean_df["Review_Text"]
    .astype(str)
    .str.lower()
    .str.strip()
    .str.replace(r"\s+", " ", regex=True)
)

duplicate_group_summary = (
    clean_df
    .groupby("Review_Normalized")
    .agg(
        group_size=("Review_Text", "size"),
        unique_scores=("Score_Numeric", "nunique"),
        unique_sentiments=("Sentiment", "nunique")
    )
    .reset_index()
)

duplicate_group_summary = duplicate_group_summary[
    duplicate_group_summary["group_size"] > 1
].copy()

conflicting_sentiment_review_keys = duplicate_group_summary[
    duplicate_group_summary["unique_sentiments"] > 1
]["Review_Normalized"]

clean_df = clean_df[
    ~clean_df["Review_Normalized"].isin(conflicting_sentiment_review_keys)
].copy()

clean_df = (
    clean_df
    .sort_values(["Review_Normalized", "Score_Numeric"])
    .drop_duplicates(subset=["Review_Normalized"], keep="first")
    .copy()
)

# Same combined stratification label and split
clean_df["Stratify_Label"] = (
    clean_df["Language"].astype(str)
    + "_"
    + clean_df["Sentiment"].astype(str)
)

train_df, temp_df = train_test_split(
    clean_df,
    test_size=0.30,
    random_state=SEED,
    stratify=clean_df["Stratify_Label"]
)

val_df, test_df = train_test_split(
    temp_df,
    test_size=0.50,
    random_state=SEED,
    stratify=temp_df["Stratify_Label"]
)

X_train_text = train_df["Review_Text"].astype(str).values
X_val_text = val_df["Review_Text"].astype(str).values
X_test_text = test_df["Review_Text"].astype(str).values

y_train = train_df["Sentiment_Encoded"].values
y_val = val_df["Sentiment_Encoded"].values
y_test = test_df["Sentiment_Encoded"].values

print("Cleaned rows:", len(clean_df))
print("Train rows:", len(train_df))
print("Validation rows:", len(val_df))
print("Test rows:", len(test_df))

assert len(clean_df) == 766, f"Expected 766 cleaned rows, got {len(clean_df)}"
assert len(train_df) == 536, f"Expected 536 train rows, got {len(train_df)}"
assert len(val_df) == 115, f"Expected 115 validation rows, got {len(val_df)}"
assert len(test_df) == 115, f"Expected 115 test rows, got {len(test_df)}"


Cleaned rows: 766
Train rows: 536
Validation rows: 115
Test rows: 115


In [6]:
# Recreate selected Strategy B language-token representation and tokenizer

def add_language_token(text_array, language_array):
    tokenised_text = []

    for text, language in zip(text_array, language_array):
        if language == "English":
            language_token = "lang_english"
        elif language == "Malay":
            language_token = "lang_malay"
        else:
            language_token = "lang_unknown"

        tokenised_text.append(f"{language_token} {text}")

    return np.asarray(tokenised_text).astype(str)

section6_train_language = train_df["Language"].to_numpy()
section6_val_language = val_df["Language"].to_numpy()
section6_test_language = test_df["Language"].to_numpy()

selected_train_text = add_language_token(X_train_text, section6_train_language)
selected_val_text = add_language_token(X_val_text, section6_val_language)
selected_test_text = add_language_token(X_test_text, section6_test_language)

selected_tokenizer = Tokenizer(
    num_words=VOCAB_SIZE,
    oov_token="<OOV>"
)
selected_tokenizer.fit_on_texts(selected_train_text)

selected_X_train = pad_sequences(
    selected_tokenizer.texts_to_sequences(selected_train_text),
    maxlen=FINAL_SEQUENCE_LENGTH,
    padding="post",
    truncating="post"
)

selected_X_val = pad_sequences(
    selected_tokenizer.texts_to_sequences(selected_val_text),
    maxlen=FINAL_SEQUENCE_LENGTH,
    padding="post",
    truncating="post"
)

selected_X_test = pad_sequences(
    selected_tokenizer.texts_to_sequences(selected_test_text),
    maxlen=FINAL_SEQUENCE_LENGTH,
    padding="post",
    truncating="post"
)

print("Selected Strategy B arrays prepared.")
print("Train shape:", selected_X_train.shape, y_train.shape)
print("Validation shape:", selected_X_val.shape, y_val.shape)
print("Test shape:", selected_X_test.shape, y_test.shape)


Selected Strategy B arrays prepared.
Train shape: (536, 15) (536,)
Validation shape: (115, 15) (115,)
Test shape: (115, 15) (115,)


In [7]:
# Rebuild final selected Part B model architecture
# Load weights before compiling to avoid irrelevant optimizer-state warnings.

tf.keras.backend.clear_session()

def build_final_part_b_model():
    model = Sequential([
        Input(shape=(FINAL_SEQUENCE_LENGTH,)),
        Embedding(
            input_dim=VOCAB_SIZE,
            output_dim=EMBEDDING_DIM,
            mask_zero=True
        ),
        Bidirectional(
            SimpleRNN(
                64,
                recurrent_dropout=0.2
            )
        ),
        Dropout(0.3),
        Dense(3, activation="softmax")
    ], name="final_rnn_weights_verification")

    return model

final_model = build_final_part_b_model()
final_model.load_weights(WEIGHTS_PATH)

final_model.compile(
    optimizer=Adam(learning_rate=0.001),
    loss="sparse_categorical_crossentropy",
    metrics=["accuracy"]
)

print("Weights loaded successfully.")
print("Weights file:", WEIGHTS_PATH)
final_model.summary()


Weights loaded successfully.
Weights file: final_rnn.weights.h5


Model: "final_rnn_weights_verification"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┓
┃ Layer (type)                    ┃ Output Shape           ┃       Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━┩
│ embedding (Embedding)           │ (None, 15, 32)         │        32,000 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ bidirectional (Bidirectional)   │ (None, 128)            │        12,416 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dropout (Dropout)               │ (None, 128)            │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense (Dense)                   │ (None, 3)              │           387 │
└─────────────────────────────────┴────────────────────────┴───────────────┘

 Total params: 44,803 (175.01 KB)

 Trainable params: 44,803 (175.01 KB)

 Non-trainable params: 0 (0.00 B)

In [8]:
# Evaluate validation and test metrics from loaded weights
val_loss, val_acc_keras = final_model.evaluate(selected_X_val, y_val, verbose=0)
test_loss, test_acc_keras = final_model.evaluate(selected_X_test, y_test, verbose=0)

val_proba = final_model.predict(selected_X_val, verbose=0)
test_proba = final_model.predict(selected_X_test, verbose=0)

val_pred = np.argmax(val_proba, axis=1)
test_pred = np.argmax(test_proba, axis=1)

val_acc = accuracy_score(y_val, val_pred)
val_macro_f1 = f1_score(y_val, val_pred, average="macro", zero_division=0)
val_weighted_f1 = f1_score(y_val, val_pred, average="weighted", zero_division=0)

test_acc = accuracy_score(y_test, test_pred)
test_macro_f1 = f1_score(y_test, test_pred, average="macro", zero_division=0)
test_weighted_f1 = f1_score(y_test, test_pred, average="weighted", zero_division=0)

print("Validation metrics from loaded weights:")
print("Validation loss:", round(val_loss, 4))
print("Validation accuracy:", round(val_acc, 4))
print("Validation macro F1:", round(val_macro_f1, 4))
print("Validation weighted F1:", round(val_weighted_f1, 4))

print("Test metrics from loaded weights:")
print("Test loss:", round(test_loss, 4))
print("Test accuracy:", round(test_acc, 4))
print("Test macro F1:", round(test_macro_f1, 4))
print("Test weighted F1:", round(test_weighted_f1, 4))

print("Expected approximate final notebook metrics:")
print("Validation: loss 0.2599, accuracy 0.9043, macro F1 0.8990, weighted F1 0.9055")
print("Test:       loss 0.2672, accuracy 0.9043, macro F1 0.9018, weighted F1 0.9032")


Validation metrics from loaded weights:
Validation loss: 0.2599
Validation accuracy: 0.9043
Validation macro F1: 0.899
Validation weighted F1: 0.9055
Test metrics from loaded weights:
Test loss: 0.2672
Test accuracy: 0.9043
Test macro F1: 0.9018
Test weighted F1: 0.9032
Expected approximate final notebook metrics:
Validation: loss 0.2599, accuracy 0.9043, macro F1 0.8990, weighted F1 0.9055
Test:       loss 0.2672, accuracy 0.9043, macro F1 0.9018, weighted F1 0.9032


In [9]:
# Optional: classification reports from loaded weights
print("Validation classification report:")
print(
    classification_report(
        y_val,
        val_pred,
        target_names=SENTIMENT_CLASSES,
        zero_division=0
    )
)

print("Test classification report:")
print(
    classification_report(
        y_test,
        test_pred,
        target_names=SENTIMENT_CLASSES,
        zero_division=0
    )
)


Validation classification report:
               precision    recall  f1-score   support

     Negative       0.93      0.84      0.88        31
Mixed/Neutral       0.80      0.94      0.87        35
     Positive       0.98      0.92      0.95        49

     accuracy                           0.90       115
    macro avg       0.90      0.90      0.90       115
 weighted avg       0.91      0.90      0.91       115

Test classification report:
               precision    recall  f1-score   support

     Negative       0.88      0.97      0.92        29
Mixed/Neutral       0.94      0.81      0.87        36
     Positive       0.90      0.94      0.92        50

     accuracy                           0.90       115
    macro avg       0.90      0.90      0.90       115
 weighted avg       0.91      0.90      0.90       115

